# Project Baseline: cohort feasibility in Python

This notebook shows how to go from a research question to an analysis-ready cohort in four steps,
using the [Project Baseline Health Study](https://www.projectbaseline.com/) (Limited Tier) data collection.

**Question:** *If we ran a study in young adults with asthma who use an asthma or airway medication,
how many participants would be eligible, and how do they compare with everyone else?*

1. Define eligibility as one SQL query, one criterion per step
2. Look at the eligibility funnel
3. Compare eligible participants with everyone else (Table 1)
4. Save the cohort to a workspace bucket

The same analysis is available in R (`cohort_feasibility_r.ipynb`) and as plain-English prompts for
Workbench AI (`workbench_ai_prompts.md`). All three give the same numbers.

**Requirements:** a workspace with the Project Baseline Health Study (Limited Tier) data collection
added, which creates the `screener`, `crf`, `analysis`, and `externallab` BigQuery resources. To save
results, the workspace also needs a bucket resource (named `uploads` below).

*The eligibility criteria are hypothetical and for illustration only.*

## Setup

The first cell makes sure the BigQuery client library and its dependencies are compatible (some app
images ship an older `google-api-core` than the BigQuery client expects). **If it installs anything,
restart the kernel (Kernel → Restart Kernel) and run the notebook again from the top.**

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade",
     "google-api-core>=2.41.0", "google-cloud-bigquery>=3.46.1", "db-dtypes>=1.4.3"],
    capture_output=True, text=True,
)
if "Successfully installed" in result.stdout:
    print("Upgraded:", result.stdout.rsplit("Successfully installed", 1)[1].strip())
    print("Restart the kernel (Kernel > Restart Kernel), then run the notebook again from the top.")
elif result.returncode == 0:
    print("Dependencies already satisfied.")
else:
    print(result.stderr.strip())

In [ ]:
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client()


def resolve(resource_name):
    """Return the cloud path of a workspace resource, e.g. 'crf' -> 'my-project.crf'."""
    return subprocess.run(["wb", "resolve", "--name", resource_name],
                          capture_output=True, text=True, check=True).stdout.strip()


SCREENER, CRF, ANALYSIS, EXTERNALLAB = (resolve(r) for r in ["screener", "crf", "analysis", "externallab"])
OUTPUT_BUCKET_RESOURCE = "uploads"  # change to any bucket resource in your workspace
print(CRF)

## Step 1: Define eligibility in one query

Each `WITH` block is one criterion, so the logic is easy to read, audit, and change. The final
`SELECT` returns one row per participant with a true/false column per criterion, plus a few
characteristics for Table 1.

Tip: in `crf.CM`, a drug that belongs to several ATC classes lists all of them in one field
(for example `G02CA:... || R03AC:...`), so match codes with `REGEXP_CONTAINS` rather than `=`.

In [ ]:
COHORT_SQL = f"""
WITH
demographics AS (
  SELECT USUBJID, SAFE_CAST(age_at_enrollment AS INT64) AS age, SEX AS sex, RACE AS race
  FROM `{SCREENER}.DM`),
asthma AS (        -- medical history includes asthma
  SELECT DISTINCT USUBJID FROM `{CRF}.MH` WHERE PTName = 'Asthma'),
airway_med AS (    -- any drug for obstructive airway disease (ATC R03), relievers and controllers
  SELECT DISTINCT USUBJID FROM `{CRF}.CM` WHERE REGEXP_CONTAINS(ATC_Codes, 'R03')),
copd AS (          -- medical history includes COPD
  SELECT DISTINCT USUBJID FROM `{CRF}.MH` WHERE PTName = 'Chronic obstructive pulmonary disease'),
retained AS (      -- attended at least 3 annual visits
  SELECT USUBJID FROM `{CRF}.VISDT1`
  WHERE VISIT LIKE '%ANNUAL_VISIT' GROUP BY USUBJID HAVING COUNT(DISTINCT VISIT) >= 3),
bmi AS (
  SELECT USUBJID, AVG(SAFE_CAST(vs_weight_kg AS FLOAT64) / POW(SAFE_CAST(vs_height_cm AS FLOAT64) / 100, 2)) AS bmi
  FROM `{CRF}.VS` WHERE VISIT = 'STUDY_START' GROUP BY USUBJID),
phq9 AS (          -- depression symptom score, first study year
  SELECT USUBJID, AVG(PHQ9_SUM_SCORE) AS phq9
  FROM `{ANALYSIS}.PHQ9A_SCORES` WHERE VISITNUM < 2 GROUP BY USUBJID),
crp AS (           -- high-sensitivity C-reactive protein at enrollment
  SELECT USUBJID, AVG(SAFE_CAST(siurslt AS FLOAT64)) AS hs_crp
  FROM `{EXTERNALLAB}.CLABS` WHERE VISIT = 'STUDY_START' AND rtcname = 'CRP HIGH SENSITIVITY' GROUP BY USUBJID)
SELECT
  d.*,
  d.USUBJID IN (SELECT USUBJID FROM asthma)         AS has_asthma,
  d.age BETWEEN 18 AND 40                           AS age_18_40,
  d.USUBJID IN (SELECT USUBJID FROM airway_med)     AS on_airway_med,
  d.USUBJID NOT IN (SELECT USUBJID FROM copd)       AS no_copd,
  d.USUBJID IN (SELECT USUBJID FROM retained)       AS retained,
  bmi.bmi, phq9.phq9, crp.hs_crp
FROM demographics d
LEFT JOIN bmi USING (USUBJID)
LEFT JOIN phq9 USING (USUBJID)
LEFT JOIN crp USING (USUBJID)
ORDER BY USUBJID
"""
participants = client.query(COHORT_SQL).to_dataframe()
participants.head()

## Step 2: Eligibility funnel

Apply the criteria one at a time and count who is left after each.

In [ ]:
criteria = {
    "Asthma": "has_asthma",
    "Age 18-40": "age_18_40",
    "On an airway medication (R03)": "on_airway_med",
    "No COPD": "no_copd",
    "3+ annual visits": "retained",
}
eligible = pd.Series(True, index=participants.index)
funnel = [("All participants", len(participants))]
for label, column in criteria.items():
    eligible &= participants[column].fillna(False)
    funnel.append((label, int(eligible.sum())))
participants["eligible"] = eligible

funnel = pd.DataFrame(funnel, columns=["criterion", "participants"])
ax = funnel[::-1].plot.barh(x="criterion", y="participants", legend=False, figsize=(7, 3))
ax.bar_label(ax.containers[0], padding=3)
ax.set_xlabel("participants")
ax.set_ylabel("")
plt.tight_layout()
funnel

## Step 3: Table 1, eligible vs. everyone else

In [ ]:
def median_iqr(values):
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    return f"{median:.1f} [{q1:.1f}-{q3:.1f}]"


def summarize(group):
    return pd.Series({
        "N": len(group),
        "Age, median [IQR]": median_iqr(group.age),
        "Female, %": round(100 * (group.sex == "Female").mean()),
        "Non-White, %": round(100 * (group.race != "White").mean()),
        "BMI, median [IQR]": median_iqr(group.bmi),
        "PHQ-9, median [IQR]": median_iqr(group.phq9),
        "hs-CRP mg/L, median [IQR]": median_iqr(group.hs_crp),
    })


table1 = participants.groupby("eligible").apply(summarize).T
table1.columns = ["Not eligible", "Eligible"]
table1[["Eligible", "Not eligible"]]

## Step 4: Save the cohort to a workspace bucket

Save the cohort, Table 1, and the exact SQL that defined them, so a collaborator can rerun or audit the
analysis later. Everything stays inside the workspace.

In [ ]:
cohort = participants[participants.eligible]
cohort.to_csv("cohort.csv", index=False)
table1.to_csv("table1.csv")
with open("cohort_definition.sql", "w") as f:
    f.write(COHORT_SQL)

bucket = resolve(OUTPUT_BUCKET_RESOURCE)
destination = f"{bucket}/cohort_feasibility_python/"
subprocess.run(["gcloud", "storage", "cp", "cohort.csv", "table1.csv", "cohort_definition.sql", destination],
               check=True, capture_output=True)
print(f"Saved {len(cohort)} participants to {destination}")

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd